<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Insights/blob/main/00%20Whisper_MP3_to_Text_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI Whisper 기반 MP3 → Text Script 변환

이 노트북은 Google Colab에서 실행할 수 있습니다.

생성 파일:
- 일반 스크립트 TXT
- 타임스탬프 포함 TXT
- SRT 자막
- Whisper 전체 결과 JSON


## 1. GPU 런타임 확인

Colab 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 권장합니다.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('사용 장치:', device)

if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU를 사용할 수 없습니다. CPU에서도 실행할 수 있지만 시간이 오래 걸릴 수 있습니다.')


## 2. Whisper와 FFmpeg 설치

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q -U openai-whisper

print('설치 완료')


## 3. MP3 파일 업로드

실행 후 변환할 MP3, WAV, M4A 등의 음성 파일을 선택합니다.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

if not uploaded:
    raise RuntimeError('업로드된 파일이 없습니다.')

audio_filename = next(iter(uploaded.keys()))
audio_path = Path(audio_filename)

print('업로드 파일:', audio_path)


## 4. Whisper 옵션 설정

- `MODEL_NAME`: 정확도와 실행 속도 조절
- `LANGUAGE`: 자동 감지는 `None`, 영어는 `en`, 한국어는 `ko`
- `TASK`: 원문 전사는 `transcribe`, 영어 번역은 `translate`


In [ ]:
# 모델 선택: tiny, base, small, medium, large, turbo
MODEL_NAME = 'medium'

# None이면 언어 자동 감지
LANGUAGE = None

# transcribe: 원래 언어로 전사
# translate: 음성을 영어 텍스트로 번역
TASK = 'transcribe'

# 인명, 전문용어 등 추가 문맥이 필요하면 문자열 입력
INITIAL_PROMPT = None

print('모델:', MODEL_NAME)
print('언어:', LANGUAGE if LANGUAGE else '자동 감지')
print('작업:', TASK)


## 5. Whisper 음성 인식 실행

In [ ]:
import whisper
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
fp16 = device == 'cuda'

print('Whisper 모델 로딩 중...')
model = whisper.load_model(MODEL_NAME, device=device)

print('음성 인식 중...')
result = model.transcribe(
    str(audio_path),
    language=LANGUAGE,
    task=TASK,
    fp16=fp16,
    verbose=False,
    initial_prompt=INITIAL_PROMPT,
    condition_on_previous_text=True,
)

print('감지 언어:', result.get('language', 'unknown'))
print('\n--- 전사 결과 미리보기 ---\n')
print(result['text'][:3000])


## 6. TXT, SRT, JSON 결과 저장

In [ ]:
import json
from pathlib import Path

def format_timestamp(seconds: float, srt: bool = False) -> str:
    milliseconds = max(0, round(seconds * 1000))
    hours, milliseconds = divmod(milliseconds, 3_600_000)
    minutes, milliseconds = divmod(milliseconds, 60_000)
    secs, milliseconds = divmod(milliseconds, 1_000)
    separator = ',' if srt else '.'
    return f'{hours:02d}:{minutes:02d}:{secs:02d}{separator}{milliseconds:03d}'

output_dir = Path('/content/whisper_results')
output_dir.mkdir(parents=True, exist_ok=True)

stem = audio_path.stem
plain_text_path = output_dir / f'{stem}_transcript.txt'
timestamped_path = output_dir / f'{stem}_timestamped.txt'
srt_path = output_dir / f'{stem}.srt'
json_path = output_dir / f'{stem}_result.json'

plain_text_path.write_text(result['text'].strip() + '\n', encoding='utf-8')

with timestamped_path.open('w', encoding='utf-8') as file:
    for segment in result.get('segments', []):
        start = format_timestamp(float(segment['start']))
        end = format_timestamp(float(segment['end']))
        text = segment['text'].strip()
        file.write(f'[{start} --> {end}] {text}\n')

with srt_path.open('w', encoding='utf-8') as file:
    for index, segment in enumerate(result.get('segments', []), start=1):
        start = format_timestamp(float(segment['start']), srt=True)
        end = format_timestamp(float(segment['end']), srt=True)
        text = segment['text'].strip()
        file.write(f'{index}\n{start} --> {end}\n{text}\n\n')

json_path.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

print('저장 완료')
print(plain_text_path)
print(timestamped_path)
print(srt_path)
print(json_path)


## 7. 결과 파일 ZIP 다운로드

In [ ]:
import shutil
from google.colab import files

zip_base = f'/content/{audio_path.stem}_whisper_results'
zip_path = shutil.make_archive(zip_base, 'zip', output_dir)

print('다운로드 파일:', zip_path)
files.download(zip_path)


## 선택 사항: 개별 TXT 파일만 다운로드


In [ ]:
# 일반 스크립트 TXT 파일만 다운로드하려면 아래 줄의 주석을 제거합니다.
# files.download(str(plain_text_path))
